# 05 — A learned ranker (LightGBM)

**The idea.** Until now, candidates were ranked with a **hand-tuned formula**: analog score (power 16) plus an identity bonus above a threshold. A **learned ranker** instead receives a set of clues (features) for each candidate and **learns** how to combine them from examples where the right answer is known.

**Learning to rank.** This is not ordinary "right answer or not" classification: what matters is the **order** of the candidates **within each molecule**. LightGBM has a dedicated objective for this (`lambdarank`): rows are grouped by molecule, and it directly optimises a ranking quality measure (NDCG, close to our MRR: both mostly reward correct answers placed at the top).

**Candidates.** As chosen in notebook 04: **COCONUT** structures of the right mass, plus any structure confirmed by a **strong identity match** (similarity ≥ 0.9) even outside COCONUT. Analogs exclude molecules found **only** in synthetic libraries (`enveda-180`, `drug_plus`).

**Data.**
- **Training**: 2,000 natural products from the reference library, **disjoint** from validation and prepared exactly like it (query spectra hidden), simulated as class 1 and class 2 (`scripts/build_ranker_set.py`).
- **Evaluation**: our usual validation (subsets A and B), draws 0 and 1, conditions class 1 and class 2.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb

import casmi

pd.set_option("display.width", 160)

## 1. Features

One row per (query molecule, candidate) pair. `label` = 1 if the candidate is the right answer.

| Feature | Meaning |
|---|---|
| `an16`, `an4`, `an1` | analog propagation score at powers 16, 4 and 1 (more or less "selective") |
| `an16_rel`, `an16_rank` | the same, relative to the molecule's best candidate, and its rank |
| `id_max`, `id_n_spectra` | best **identity** similarity (same mass) and number of query spectra that retrieve this candidate |
| `tani_best_analog`, `tani_max_top10` | structural similarity (Tanimoto) to the best analog, and to the closest of the top 10 |
| `has_visible_spectra` | does the candidate have (non-hidden) reference spectra? |
| `in_coconut`, `np_likeness` | present in COCONUT; natural-product likeness score |
| `mass_err_ppm`, `exact_mass` | gap between candidate mass and measured mass; mass |
| `n_candidates`, `best_analog_sim`, `n_query_spectra` | **molecule-level** features (same for all its candidates): how hard the case is |

**Leakage trap, avoided.** `has_visible_spectra` is computed on the references **allowed** in the simulated condition, not on the whole library. Otherwise, in class 2, the simulated right answer "would have spectra" (we only hid them), whereas a real class-2 molecule never does: the ranker would learn a false rule.

**Opposite effect depending on the class.** In class 1, the right answer has visible spectra; in class 2, never. The ranker learns a single compromise that depends on the **mix** of classes in training, which should resemble the test (~16 % class 1 vs ~45 % class 2). The `condition` column is used for weighting only: it is **not** a feature, since on the test the model cannot know which class it is facing.

In [2]:
pool = casmi.candidate_pool()
fps = casmi.pool_fingerprints(pool)
np_like = casmi.pool_np_likeness(pool)

meta = casmi.load_reference_meta()
counts = casmi.spectra_count_distribution()
# Pool chosen in notebook 04: COCONUT candidates (+ strong identity), analogs without synthetic-only molecules
candidate_mask, analog_mask = casmi.pool_masks(pool, meta)


def feature_table(name, structures_df, queries_df, draw, cond):
    # Build (or reload from cache) the feature table of a query set, for one condition
    def compute():
        allowed = casmi.reference_mask(meta, queries_df, cond)
        visible = set(meta.loc[allowed, "key"])
        truth = structures_df.set_index("molecule_id")["key"].to_dict()
        return casmi.candidate_features(
            casmi.cached(f"hits_identity_{name}", None), casmi.cached(f"hits_hybrid_{name}", None),
            draw, pool, fps, visible, np_like, truth=truth,
            candidate_mask=candidate_mask, analog_mask=analog_mask)
    return casmi.cached(f"features_v2_{name}", compute).assign(condition=cond)


# Training: ranker set, three simulated conditions (class1 and class1x are two versions of class 1)
rank_structures = pd.read_parquet(casmi.paths.processed / "rank_structures.parquet")
rank_queries = pd.read_parquet(casmi.paths.processed / "rank_queries.parquet")
rank_draw = pd.read_parquet(casmi.paths.processed / "rank_draw.parquet")
train = pd.concat([feature_table(f"{c}_rank", rank_structures, rank_queries, rank_draw, c)
                   for c in ["class1", "class1x", "class2"]], ignore_index=True)

# Validation: A and B, draws 0 and 1, conditions class 1 and class 2
val_structures, val_queries = casmi.load_validation()
subset = val_structures.set_index("molecule_id")["subset"]
val = []
for seed in [0, 1]:
    draw = casmi.sample_queries(val_queries, counts, seed)
    for c in ["class1", "class2"]:
        val.append(feature_table(f"{c}_s{seed}", val_structures, val_queries, draw, c).assign(seed=seed))
val = pd.concat(val, ignore_index=True)
val["subset"] = val["molecule_id"].map(subset)

print(f"training  : {len(train):,} rows, {train.groupby(['condition', 'molecule_id']).ngroups:,} groups")
print(f"validation: {len(val):,} rows")
train.head()

training  : 409,096 rows, 5,997 groups
validation: 366,110 rows


,molecule_id,key,an16,an4,an1,an16_rel,an16_rank,id_max,id_n_spectra,tani_best_analog,...,has_visible_spectra,in_coconut,np_likeness,exact_mass,mass_err_ppm,n_candidates,best_analog_sim,n_query_spectra,label,condition
0,t_0000,BHDBRCJCCXGBOF,8.857136,9.892993,15.129219,0.421404,10.0,0.000000,0,0.135417,...,0,1,1.14,818.402998,7.333673,13,1.0,6,0,class1
1,t_0000,KZYJZNHECRRWEC,7.307740,8.012718,11.794030,0.347687,12.0,0.000000,0,0.112069,...,0,1,1.41,818.402998,7.333673,13,1.0,6,0,class1
2,t_0000,WPCZHBSWUQTPRF,7.376896,8.209504,12.932491,0.350977,11.0,0.000000,0,0.086614,...,0,1,-0.14,818.407523,1.804345,13,1.0,6,0,class1
3,t_0000,YEVBCZUKBWBKQX,21.018164,24.656694,41.593281,1.000000,1.0,1.000000,6,0.238532,...,1,1,2.67,818.408871,0.157276,13,1.0,6,0,class1
4,t_0000,ABGZLUPZDXEGLQ,20.687366,24.211613,40.829243,0.984261,2.0,0.641846,5,0.238532,...,1,1,2.60,818.408871,0.157276,13,1.0,6,1,class1


## 2. Evaluation

`evaluate_table(val, column)` ranks each molecule's candidates by a score column, then computes MRR@25 per condition and subset, and the **expected score** = 0.16 × class 1 + 0.45 × class 2 (class 3 scores 0 with this approach, so the maximum is 0.61). This is the number to compare between experiments.

Baseline: the current formula (submission 1, with the settings from notebook 04).

In [3]:
def evaluate_table(table, score_col):
    rows = []
    for (cond, seed), g in table.groupby(["condition", "seed"]):
        ev = casmi.evaluate(casmi.scores_from_table(g, score_col), val_structures)
        for sub, e in ev.groupby("subset"):
            rows.append(dict(condition=cond, seed=seed, subset=sub, mrr=e["rr"].mean()))
    r = pd.DataFrame(rows).groupby(["subset", "condition"])["mrr"].mean().unstack("condition")
    r["expected"] = 0.16 * r["class1"] + 0.45 * r["class2"]
    return r.round(4)


# Current formula: normalised analog score + 10 x identity when it exceeds 0.9
val["formula"] = val["an16_rel"] + np.where(val["id_max"] >= 0.9, 10 * val["id_max"], 0.0)
evaluate_table(val, "formula")

condition,class1,class2,expected
subset,,,
A_np_examples,0.8794,0.6117,0.4160
B_np_public,0.7258,0.5484,0.3629


## 3. Training the ranker

`LGBMRanker.fit(X, y, group=sizes, sample_weight=w)`:

- **Groups.** `group` is the **size** of each group (one molecule in one condition), **in row order**: rows are sorted by (condition, molecule) and sizes computed with `groupby(..., sort=False).size()`. LightGBM only compares candidates within the same group.
- **Weights.** One weight per row; in ranking mode LightGBM turns them into one weight per group. The weight of a class-1 group is proportional to (test share of class 1) / (number of class-1 groups), so that the total weight of each class matches the test mix (16 : 45), whatever the number of groups.
- **Same molecule in both classes.** Each training molecule appears once as a class-1 query and once as a class-2 query. This doubles the data and is harmless: the model never sees the molecule's identity, only features, which differ between the two situations.

The ranker is built in four steps, each fixing a problem revealed by the previous one:

1. **Ranker, class 1**: class-1 queries simulated by hiding only the query's own source (same library **and** same instrument family).
2. **Class 1x**: hide the whole instrument family, in every library. The hidden test is measured on timsTOF, and no public library contains timsTOF spectra of natural products, so a real class-1 match is always **cross-instrument**.
3. **Reachable filter**: drop class-1 groups whose right answer has no visible spectrum left (the molecule was only measured in the hidden family). Such groups are class 2 in disguise and teach the model that the right answer often has no identity match.
4. **Rebalanced weights**: the filter removes about half of the class-1 groups, so per-group weights are recomputed to restore the 16 : 45 mix (steps 1–3 use 0.16 and 0.45 per group).

In [4]:
CLASS_SHARE = pd.Series({"class1": 0.16, "class2": 0.45})  # estimated shares of each class in the test


def prepare(class1_condition, keep_reachable):
    # Training table: one version of class 1 + class 2, sorted so that each group is contiguous
    t = train[train["condition"].isin([class1_condition, "class2"])].copy()
    t["condition"] = t["condition"].replace({class1_condition: "class1"})
    if keep_reachable:
        # Keep a class-1 group only if its right answer still has visible reference spectra
        reachable = set(t.loc[(t["condition"] == "class1") & (t["label"] == 1)
                              & (t["has_visible_spectra"] == 1), "molecule_id"])
        t = t[(t["condition"] == "class2") | t["molecule_id"].isin(reachable)]
    return t.sort_values(["condition", "molecule_id"])


def fit_ranker(t, rebalance, n_estimators=100):
    group_sizes = t.groupby(["condition", "molecule_id"], sort=False).size()
    assert group_sizes.sum() == len(t)
    n_groups = t.groupby("condition")["molecule_id"].nunique()
    group_weight = CLASS_SHARE / n_groups if rebalance else CLASS_SHARE.copy()
    group_weight = group_weight / group_weight.mean()  # keep weights around 1 (LightGBM thresholds are sums of weights)
    model = lgb.LGBMRanker(objective="lambdarank", n_estimators=n_estimators, verbose=-1)
    model.fit(t[casmi.FEATURES], t["label"], group=group_sizes,
              sample_weight=t["condition"].map(group_weight))
    return model


STEPS = {
    "1. ranker, class 1": dict(class1_condition="class1", keep_reachable=False, rebalance=False),
    "2. class 1x": dict(class1_condition="class1x", keep_reachable=False, rebalance=False),
    "3. reachable filter": dict(class1_condition="class1x", keep_reachable=True, rebalance=False),
    "4. rebalanced weights": dict(class1_condition="class1x", keep_reachable=True, rebalance=True),
}

results = {"formula": evaluate_table(val, "formula")}
class1_groups = {}
for name, s in STEPS.items():
    t = prepare(s["class1_condition"], s["keep_reachable"])
    class1_groups[name] = t.loc[t["condition"] == "class1", "molecule_id"].nunique()
    model = fit_ranker(t, s["rebalance"])
    val[name] = model.predict(val[casmi.FEATURES])
    results[name] = evaluate_table(val, name)

print("class-1 training groups per step:", class1_groups)
comparison = pd.concat({name: r.stack() for name, r in results.items()}, axis=1)
comparison

class-1 training groups per step: {'1. ranker, class 1': 1999, '2. class 1x': 1999, '3. reachable filter': 992, '4. rebalanced weights': 992}


formula  1. ranker, class 1  2. class 1x  3. reachable filter  4. rebalanced weights
subset        condition                                                                                      
A_np_examples class1      0.8794              0.6764       0.7295               0.7559                 0.8338
              class2      0.6117              0.6918       0.6939               0.6750                 0.6616
              expected    0.4160              0.4195       0.4290               0.4247                 0.4311
B_np_public   class1      0.7258              0.7559       0.7573               0.7627                 0.7765
              class2      0.5484              0.6217       0.6197               0.6173                 0.5927
              expected    0.3629              0.4007       0.4000               0.3998                 0.3910

## 4. What the model learned

Feature importance of the final model (step 4). *Gain* is the total improvement of the objective brought by the splits on a feature; *splits* is how often the feature is used.

In [5]:
importance = pd.DataFrame({
    "gain_share": model.booster_.feature_importance(importance_type="gain"),
    "splits": model.booster_.feature_importance(importance_type="split"),
}, index=casmi.FEATURES)
importance["gain_share"] /= importance["gain_share"].sum()
importance.sort_values("gain_share", ascending=False).round(3)

,gain_share,splits
an16_rel,0.530,300
tani_max_top10,0.081,387
an16_rank,0.063,275
exact_mass,0.063,190
np_likeness,0.055,441
tani_best_analog,0.037,303
mass_err_ppm,0.032,217
an4,0.026,136
id_max,0.023,127
an1,0.022,157


## 5. Conclusions

**Step 1: the ranker beats the formula everywhere except on class 1 of subset A, where it collapses** (0.68 vs 0.88). The cause is the way class 1 was simulated. Hiding only the query's own source leaves visible, in other libraries, spectra of the same molecule measured on the same instrument family, sometimes copies of the same measurement. In training, a right answer therefore typically has an identity similarity close to 0.99. Subset A is measured on timsTOF, and no reference library contains timsTOF spectra of these natural products: its class-1 matches are cross-instrument, with a median similarity around 0.90. The model learned that such scores mean "close analog, not the molecule itself" and ranks the right answer too low.

**Step 2: simulating class 1 as cross-instrument (class 1x)** brings training closer to the test: 0.68 → 0.73 on A, class 1. It has a side effect: half of the class-1 groups no longer have any visible spectrum of the right answer (the molecule had only been measured on the hidden instrument family), against 40 % before.

**Step 3: the reachable filter** removes those groups, which are class 2 in disguise: 0.73 → 0.76.

**Step 4: rebalancing the weights.** The filter halved the number of class-1 groups, and with it the total weight of class 1. Restoring the 16 : 45 mix raises class 1 markedly (0.76 → 0.83 on A) at a small cost on class 2. This is expected: there is a single scoring function for both classes, and some candidates look the same but carry opposite labels depending on the class. A candidate with visible spectra and an identity similarity of 0.85 is often the right answer in class 1, and always a decoy in class 2. The weights act as a prior on which situation is more likely, and decide how such ambiguous cases are ranked.

**Overall**, the steps gain on A (expected score 0.4195 → 0.4311) and lose on B (0.4007 → 0.3910): they trade class 2 for class 1. This is defensible because the hidden test (timsTOF natural products) resembles A more than B. The differences on A are, however, of the order of the noise (about 250 molecules), so they confirm the direction rather than measure the gain precisely.

**What the model uses.** The relative analog score `an16_rel` carries more than half of the gain: the learned ranker is first of all a refined version of the formula's analog term. The identity similarity `id_max` brings only about 2 % of the gain, which is consistent with the remaining weakness on class 1 of A. `exact_mass` (6 %) deserves caution: it may capture properties of the training set's mass distribution rather than a general rule.

**Remaining gap and next steps.**
- Even the full version in `scripts/train_ranker.py` (same steps, plus derived features and a number of trees chosen by grouped cross-validation) stays below the formula on class 1 of A (0.78 vs 0.88). The submitted model therefore adds subset A itself to training (weight 3), the only data showing real timsTOF queries, and keeps an identity guard (similarity ≥ 0.97 always ranks first). Its class-1 score on A, measured by cross-validation, is 0.86.
- The number of trees (100, LightGBM's default) was not tuned here. It should be chosen on held-out training molecules (grouped split or `GroupKFold` by molecule), not on the validation set.